[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ohtopico/site/blob/main/notebooks/wikipedia/05-texto-e-modelos-de-linguagem.ipynb)

# O texto dos verbetes e como um modelo de linguagem prevê a próxima palavra

Modelos como o ChatGPT aprendem com bilhões de textos, e a Wikipédia é um deles. Neste notebook você baixa o texto de verbetes, conta palavras e constrói um modelo de linguagem minúsculo que prevê a próxima palavra. É a mesma ideia dos grandes modelos, em escala de sala de aula.

**Público:** professores, 8º e 9º ano, ensino médio e cursos técnicos · **Tempo:** 2 aulas de 50 min · **BNCC:** EF08CO11, EF09CO10, EF07MA36

Notebook de [Júlia Roberta Quoos Alves](https://ohtopico.com.br/curriculo/) · [ohtopico.com.br](https://ohtopico.com.br/interesse-por-ia/notebooks/) · Licença CC BY-NC-SA 4.0

**Como usar:** clique em cada célula de código e aperte o botão de play (ou Shift + Enter), de cima para baixo. Os campos ao lado das células (listas e caixas de texto) podem ser trocados à vontade: depois de mudar, rode a célula de novo.

## Objetivos
- Transformar texto em dados (palavras e frequências).
- Entender que um modelo de linguagem calcula a probabilidade da próxima palavra.
- Ver o efeito da semente (seed) e da temperatura no texto gerado.
- Discutir limites: o modelo repete padrões do texto, não sabe se algo é verdade.

Os textos da Wikipédia são publicados sob a licença CC BY-SA. Ao usar trechos, cite a Wikipédia como fonte.

In [ ]:
# @title Configuração (rode primeiro) { display-mode: "form" }
# Funções para baixar os dados públicos da Wikimedia. Não precisa mudar nada aqui.
import json, re, time, urllib.parse
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact, widgets

CORES = ['#157f3b', '#b3123a', '#c99a00', '#111111', '#5fb27c', '#7a1f33']
plt.rcParams.update({'figure.figsize': (11, 4.5), 'axes.prop_cycle': plt.cycler(color=CORES), 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .3, 'font.size': 11})
pd.options.display.float_format = '{:,.0f}'.format
UA = {'User-Agent': 'notebooks-ohtopico/1.0 (https://ohtopico.com.br; contato@ohtopico.com.br)'}
FONTE = 'Fonte: Wikimedia, visitas de pessoas à Wikipédia em português'


def _get(url, params=None):
    r = requests.get(url, params=params, headers=UA, timeout=60)
    if r.status_code == 404:
        return None
    r.raise_for_status()
    return r.json()


def titulo_oficial(titulo):
    """Corrige maiúsculas e segue redirecionamentos: 'ia' vira 'Inteligência artificial'."""
    j = _get('https://pt.wikipedia.org/w/api.php', dict(action='query', titles=titulo, redirects=1, format='json', formatversion=2))
    pagina = j['query']['pages'][0]
    if pagina.get('missing'):
        raise ValueError(f'Não achei o verbete "{titulo}" na Wikipédia em português. Confira a grafia.')
    return pagina['title']


def visitas(titulos, inicio='2016-01-01', fim=None, por='mes'):
    """Tabela com as visitas de cada verbete. por='mes' ou por='dia'."""
    if isinstance(titulos, str):
        titulos = [titulos]
    fim = pd.Timestamp(fim) if fim else pd.Timestamp.today().normalize() - pd.Timedelta(days=1)
    if por == 'mes':
        fim = fim.replace(day=1) - pd.Timedelta(days=1)  # só meses completos
    colunas = {}
    for t in titulos:
        t = titulo_oficial(t)
        art = urllib.parse.quote(t.replace(' ', '_'), safe='')
        g = 'monthly' if por == 'mes' else 'daily'
        url = (f'https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/pt.wikipedia/all-access/user/{art}/{g}/'
               f'{pd.Timestamp(inicio):%Y%m%d}00/{fim:%Y%m%d}00')
        j = _get(url) or {'items': []}
        colunas[t] = pd.Series({pd.Timestamp(i['timestamp'][:8]): i['views'] for i in j['items']}, dtype='float64')
        time.sleep(.1)
    df = pd.DataFrame(colunas).sort_index()
    idx = pd.date_range(pd.Timestamp(inicio), fim, freq='MS' if por == 'mes' else 'D')
    df = df.reindex(idx).fillna(0).astype(int)
    df.index.name = 'mês' if por == 'mes' else 'dia'
    return df


def texto(titulo):
    """Texto puro de um verbete (sem tabelas nem imagens)."""
    t = titulo_oficial(titulo)
    j = _get('https://pt.wikipedia.org/w/api.php', dict(action='query', prop='extracts', explaintext=1, titles=t,
                                                         format='json', formatversion=2))
    return j['query']['pages'][0].get('extract', '')


def fmt(v, casas=0):
    """Número no formato brasileiro: 1.234,5"""
    return f'{v:,.{casas}f}'.replace(',', '_').replace('.', ',').replace('_', '.')


def baixar_csv(df, nome):
    df.to_csv(nome)
    try:
        from google.colab import files
        files.download(nome)
    except ImportError:
        print('Arquivo salvo:', nome)


print('Pronto. Funções disponíveis: visitas(), texto(), titulo_oficial(), fmt(), baixar_csv().')

## Passo 1. Baixe os textos

In [ ]:
verbetes = 'Inteligência artificial, Aprendizado de máquina, Rede neural artificial'  # @param {type:"string"}
textos = {}
for v in [x.strip() for x in verbetes.split(',') if x.strip()]:
    t = titulo_oficial(v)
    textos[t] = texto(t)
    print(f'{t}: {fmt(len(textos[t].split()))} palavras')
print()
print(list(textos.values())[0][:500], '...')

## Passo 2. Do texto às palavras
Separamos o texto em palavras (em letras minúsculas) e tiramos as palavras muito comuns, como "de", "a" e "que", chamadas **palavras vazias**.

In [ ]:
VAZIAS = set('''a à ao aos as às com como da das de do dos e é em entre essa esse esta este eu foi há isso já mais mas na nas
no nos não o os ou para pela pelas pelo pelos por que quando se sem ser seu sua são também um uma umas uns ele ela eles elas
sobre ter tem foram pode podem muito outros outras qual quais onde até após cada isso pois sendo seja'''.split())

def palavras(t):
    return re.findall(r"[a-záàâãéêíóôõúç]+(?:-[a-záàâãéêíóôõúç]+)*", t.lower())

@interact(verbete=list(textos), quantas=widgets.IntSlider(value=15, min=5, max=30))
def frequentes(verbete, quantas):
    cont = pd.Series([p for p in palavras(textos[verbete]) if p not in VAZIAS and len(p) > 2]).value_counts().head(quantas)
    ax = cont.sort_values().plot(kind='barh', color=CORES[0], edgecolor='#111', figsize=(9, .32 * quantas + 1))
    ax.set_title(f'Palavras mais frequentes em {verbete}'); ax.set_xlabel('Vezes no texto'); plt.show()

## Passo 3. Um modelo que prevê a próxima palavra
Contamos, em todos os textos, qual palavra veio depois de cada palavra. Isso é um **modelo de bigramas**. Para prever o que vem depois de "rede", o modelo olha todas as vezes que "rede" apareceu e calcula a porcentagem de cada palavra seguinte. Os grandes modelos fazem algo parecido, mas olham para muitas palavras anteriores ao mesmo tempo e aprendem com redes neurais.

In [ ]:
from collections import Counter, defaultdict
seguintes = defaultdict(Counter)
for t in textos.values():
    ps = palavras(t)
    for a, b in zip(ps, ps[1:]):
        seguintes[a][b] += 1
print(f'O modelo conhece {fmt(len(seguintes))} palavras.')

palavra = 'rede'  # @param {type:"string"}
c = seguintes.get(palavra.lower())
if not c:
    print('Essa palavra não aparece nos textos. Tente outra.')
else:
    prob = pd.Series(c).sort_values(ascending=False) / sum(c.values()) * 100
    pd.options.display.float_format = '{:.1f}'.format
    display(prob.head(10).rename('probabilidade (%)').to_frame())

## Passo 4. Gerando texto: semente e temperatura
Para gerar um texto, o modelo sorteia a próxima palavra de acordo com as probabilidades e repete.
- A **semente** fixa o sorteio: com a mesma semente, sai sempre o mesmo texto.
- A **temperatura** muda o sorteio. Baixa (perto de 0,1): quase sempre a palavra mais provável, texto repetitivo. Alta (perto de 2): palavras improváveis aparecem mais, texto mais criativo e mais sem sentido.

In [ ]:
def gerar(inicio, n, semente, temperatura):
    rng = np.random.default_rng(semente)
    saida = [inicio.lower()]
    for _ in range(n):
        c = seguintes.get(saida[-1])
        if not c:
            break
        opcoes = list(c)
        p = np.array([c[o] for o in opcoes], dtype=float) ** (1 / temperatura)
        saida.append(rng.choice(opcoes, p=p / p.sum()))
    return ' '.join(saida)

@interact(inicio=widgets.Text(value='inteligência', description='começo'),
          palavras_geradas=widgets.IntSlider(value=25, min=5, max=60),
          semente=widgets.IntSlider(value=1, min=0, max=50),
          temperatura=widgets.FloatSlider(value=1, min=.1, max=2.5, step=.1))
def escrever(inicio, palavras_geradas, semente, temperatura):
    print(gerar(inicio, palavras_geradas, semente, temperatura))

**Para discutir:**
- O texto gerado tem cara de português, mas faz sentido? Por quê?
- O modelo pode inventar uma frase falsa que parece verdadeira? (Sim: ele só repete padrões de palavras, não confere fatos. Os grandes modelos também podem fazer isso, o que chamamos de alucinação.)
- Se o texto de treino tiver um erro ou um preconceito, o que acontece com o modelo?

## Sua vez
Troque os verbetes do Passo 1 por textos de outra área (por exemplo, *Futebol, Copa do Mundo FIFA, Seleção Brasileira de Futebol*), rode tudo de novo e gere texto a partir de "seleção". O que muda?

In [ ]:
# escreva aqui

## Para fechar
Um modelo de linguagem é, no fundo, uma máquina de calcular a probabilidade da próxima palavra. Ele fica melhor com mais texto e mais contexto, mas continua dependendo do que leu. Por isso, o que a IA escreve precisa ser conferido, como qualquer outra fonte.